In [1]:
from pathlib import Path
import pandas as pd
import folium
import osmnx as ox
import numpy as np
import networkx as nx
import osmnx as ox
import json
import os
from datetime import datetime
from pulp import (
    LpProblem, LpMinimize, LpVariable, lpSum, LpBinary,
    LpStatus, value, PULP_CBC_CMD
)

### odczytanie punktów wrocławia - szpital i laby

In [2]:
#folder do zapisów csv
OUT_DIR = Path("wroclaw_map_output_batch")
OUT_DIR.mkdir(exist_ok=True)

#folder do zapisów map
OUT_RES_DIR = Path("results_batch")
OUT_RES_DIR.mkdir(exist_ok=True)

# ============================
# 1. WCZYTANIE PUNKTÓW
# ============================

points_file = OUT_DIR / "points_manual.csv"

if points_file.exists():
    print("Punkty:", (OUT_DIR / "points_manual.csv").resolve())
    df = pd.read_csv(points_file)
else:
    raise FileNotFoundError("Brak pliku points_manual.csv")

print("Liczba punktów:", len(df))
points = df.to_dict(orient="records")


# ============================
# 2. WCZYTANIE GRAFU DROGOWEGO
# ============================

graph_file = OUT_DIR / "wroclaw_drive.graphml"

if graph_file.exists():
    print("Graf istnieje — wczytuję...")
    G = ox.load_graphml(graph_file)
else:
    raise FileNotFoundError("Brak pliku wroclaw_drive.graphml — uruchom preparing.ipynb")

print("Graf wczytany.")


# ============================
# 3. KONWERSJA GRAFU NA GDF
# ============================

nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)

print("Nodes:", len(nodes_gdf))
print("Edges:", len(edges_gdf))


# ============================
# 4. Mapa
# ============================

map_file = OUT_DIR / "wroclaw_roads_points.html"

if map_file.exists():
    print("Mapa istnieje :", map_file.resolve())
else:
    print("Brak mapy")

# ============================================================
# 5. WCZYTANIE TRASY SZPITAL → LAB
# ============================================================

with open(f"{OUT_DIR}/route.json", "r") as f:
    route_data = json.load(f)

hospital_name = route_data["hospital_name"]
lab_name = route_data["lab_name"]
hospital_node = route_data["hospital_node"]
lab_node = route_data["lab_node"]
route = route_data["route"]
route_length = route_data["route_length"]
route_coords = route_data["route_coords"]

print("\nWczytano dane trasy:")
print("Długość trasy (m):", route_length)
print("Liczba punktów na trasie:", len(route))



######### wczytanie nodes_df wraz z nearest_osm - najbliższy wezeł ##############

nodes_file = OUT_DIR / "nodes_with_osm_ids.csv"

if nodes_file.exists():
    print("\nWczytuję nodes_df z:", nodes_file.resolve())
    nodes_df = pd.read_csv(nodes_file, encoding="utf-8-sig")
else:
    raise FileNotFoundError("Brak pliku nodes.csv")


######### wczytanie labs_df ##############

labs_file = OUT_DIR / "labs.csv"

if labs_file.exists():
    print("\nWczytuję labs_df z:", labs_file.resolve())
    labs_df = pd.read_csv(labs_file, encoding="utf-8-sig")
else:
    raise FileNotFoundError("Brak pliku labs.csv")



############# wczytanie macierzy czasów przejazdu i odległości ################
time_matrix_file = OUT_DIR / "travel_time_matrix_min.csv"

if time_matrix_file.exists():
    print("\nWczytuję time_matrix_min z:", time_matrix_file.resolve())
    time_matrix = pd.read_csv(time_matrix_file, encoding="utf-8-sig", index_col=0)
else:
    raise FileNotFoundError("Brak pliku time_matrix_min.csv")


distance_matrix_file = OUT_DIR / "distance_matrix_m.csv"

if distance_matrix_file.exists():
    print("\nWczytuję distance_matrix_file z:", distance_matrix_file.resolve())
    distance_matrix = pd.read_csv(distance_matrix_file, encoding="utf-8-sig", index_col=0)
else:
    raise FileNotFoundError("Brak pliku distance_matrix_file.csv")


travel_long_file = OUT_DIR / "travel_times_long.csv"

if travel_long_file.exists():
    print("\nWczytuję travel_long_file z:", travel_long_file.resolve())
    travel_long = pd.read_csv(travel_long_file, encoding="utf-8-sig", index_col=0)
else:
    raise FileNotFoundError("Brak pliku travel_long_file.csv")


############ zmiana indeksów na int ###################3
# indeksy/kolumny jako int
time_matrix.index = time_matrix.index.astype(int)
time_matrix.columns = time_matrix.columns.astype(int)

distance_matrix.index = distance_matrix.index.astype(int)
distance_matrix.columns = distance_matrix.columns.astype(int)

Punkty: C:\Users\annad\Desktop\operations\wroclaw_map_output_batch\points_manual.csv
Liczba punktów: 41
Graf istnieje — wczytuję...
Graf wczytany.
Nodes: 7297
Edges: 17220
Mapa istnieje : C:\Users\annad\Desktop\operations\wroclaw_map_output_batch\wroclaw_roads_points.html

Wczytano dane trasy:
Długość trasy (m): 5965.053013257751
Liczba punktów na trasie: 50

Wczytuję nodes_df z: C:\Users\annad\Desktop\operations\wroclaw_map_output_batch\nodes_with_osm_ids.csv

Wczytuję labs_df z: C:\Users\annad\Desktop\operations\wroclaw_map_output_batch\labs.csv

Wczytuję time_matrix_min z: C:\Users\annad\Desktop\operations\wroclaw_map_output_batch\travel_time_matrix_min.csv

Wczytuję distance_matrix_file z: C:\Users\annad\Desktop\operations\wroclaw_map_output_batch\distance_matrix_m.csv

Wczytuję travel_long_file z: C:\Users\annad\Desktop\operations\wroclaw_map_output_batch\travel_times_long.csv


### zmienianie parametrów próbek - first set i first set cars

In [10]:
######## FIRST SET #########
requests = []

for i, row in labs_df.iterrows():
    num_samples = np.random.randint(0, 25) 
    requests.append({
        "request_id": i + 1,
        "lab_node_id": int(row["node_id"]),
        "num_samples": num_samples,    #liczba próbek do oddania przez lab
        "ready_time": 0,
        "due_time": 480,
        "service_time": 5,
        "max_transport_time": 480
    })

requests_df = pd.DataFrame(requests)


# requests_df.to_csv(f"{OUT_DIR}/requests.csv", index=False, encoding="utf-8-sig")

### first set vehicles

In [9]:
############### VEHICLES ################
#capacity to ilość batchy
vehicles = [
    {"vehicle_id": 1, "start_node": 0, "end_node": 0, "capacity": 50, "shift_start": 0, "shift_end": 600},
    {"vehicle_id": 2, "start_node": 0, "end_node": 0, "capacity": 60, "shift_start": 0, "shift_end": 600},
    {"vehicle_id": 3, "start_node": 0, "end_node": 0, "capacity": 20, "shift_start": 0, "shift_end": 600},
]


vehicles_df = pd.DataFrame(vehicles)
# vehicles_df.to_csv(f"{OUT_DIR}/vehicles.csv", index=False, encoding="utf-8-sig")

### wynik

In [11]:
from milp_model_batch import solve_vrp
from milp_visualize_batch import visualize_routes

batch_size=10 # ile próbek ma jeden batch

### uruchomienie solvera
results = solve_vrp(nodes_df, requests_df, vehicles_df, time_matrix, batch_size)

if results["status"] != "Optimal":
    print("Brak tras do wizualizacji — solver nie znalazł rozwiązania.")
else:
    print("Znaleziono rozwiązanie MILP.")


#zapis wyników do jendego folderu indeksowanego datą
RESULTS_ROOT = Path("results")

timestamp = datetime.now().strftime("%d_%m_%Y_%H-%M")
RUN_DIR = RESULTS_ROOT / timestamp
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Folder symulacji:", RUN_DIR)


#zapis request i vehicles do tego folderu 
requests_path = RUN_DIR / "requests.csv"
vehicles_path = RUN_DIR / "vehicles.csv"

requests_df.to_csv(requests_path, index=False, encoding="utf-8-sig")
vehicles_df.to_csv(vehicles_path, index=False, encoding="utf-8-sig")



#zapis wyniku
results["selected_arcs"].to_csv(
    RUN_DIR / "milp_selected_arcs.csv",
    index=False,
    encoding="utf-8-sig"
)

results["routes"].to_csv(
    RUN_DIR / "milp_routes.csv",
    index=False,
    encoding="utf-8-sig"
)

results["lab_assignments"].to_csv(
    RUN_DIR / "milp_lab_assignments.csv",
    index=False,
    encoding="utf-8-sig"
)

results["load"].to_csv(
    RUN_DIR / "milp_load.csv",
    index=False,
    encoding="utf-8-sig"
)

results["route_load"].to_csv(
    RUN_DIR / "milp_route_load.csv",
    index=False,
    encoding="utf-8-sig"
)

results["travel_time_summary"].to_csv(
    RUN_DIR / "milp_travel_time_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

results["total_route_time"].to_csv(
    RUN_DIR / "milp_total_route_time.csv",
    index=False,
    encoding="utf-8-sig"
)


print("Zapisano wyniki solvera w:", RUN_DIR)


#wizualizacja
if results["status"] == "Optimal":
    routes_df = results["routes"]

    routes_dict = (
        routes_df
        .sort_values(["vehicle_id", "stop_order"])
        .groupby("vehicle_id")["node"]
        .apply(list)
        .to_dict()
    )

    # ============================================================
    # 6. WIZUALIZACJA TRAS I ZAPIS MAPY
    # ============================================================

    summary = visualize_routes(
        nodes_df=nodes_df,
        routes=routes_dict,
        G=G,
        OUT_DIR=RUN_DIR,
        map_filename="milp_routes_map.html"
    )

    summary.to_csv(
        RUN_DIR / "route_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("Mapa i podsumowanie zapisane w:", RUN_DIR)

# ============================================================
# 7. PODSUMOWANIE
# ============================================================

print("\n=== PODSUMOWANIE SYMULACJI ===")
print("Folder:", RUN_DIR)
print("Objective:", results["objective"])
print("Status:", results["status"])

Brak tras do wizualizacji — solver nie znalazł rozwiązania.
Folder symulacji: results\16_04_2026_07-39
Zapisano wyniki solvera w: results\16_04_2026_07-39

=== PODSUMOWANIE SYMULACJI ===
Folder: results\16_04_2026_07-39
Objective: 108.37178445076921
Status: Not Solved


### second set

In [18]:
####### SECOND SET #########
requests = []

for i, row in labs_df.iterrows():
    ready_time = np.random.randint(0, 5)        # moment powstania próbki
    num_samples = np.random.randint(1, 30) 
    # due_time = ready_time + np.random.randint(0, 300)
    requests.append({
        "request_id": i + 1,
        "lab_node_id": int(row["node_id"]),
        "num_samples": num_samples,    #liczba próbek do oddania przez lab
        "ready_time": ready_time,          # można odbierać od początku
        "due_time": 600,          # szerokie okno
        "service_time": 5,
        "max_transport_time": 600 # szeroki limit
    })


requests_df = pd.DataFrame(requests)

### second set vehicles

In [19]:
####### SECOND SET OF CARS ######### ---> NOT WOKRING
vehicles = [
    {"vehicle_id": 1, "start_node": 0, "end_node": 0, "capacity": 50, "shift_start": 0, "shift_end": 600},
    {"vehicle_id": 2, "start_node": 0, "end_node": 0, "capacity": 20, "shift_start": 0, "shift_end": 600},
    {"vehicle_id": 3, "start_node": 0, "end_node": 0, "capacity": 20, "shift_start": 0, "shift_end": 600},
    {"vehicle_id": 4, "start_node": 0, "end_node": 0, "capacity": 20, "shift_start": 0, "shift_end": 600},
    {"vehicle_id": 5, "start_node": 0, "end_node": 0, "capacity": 50, "shift_start": 0, "shift_end": 600},
    {"vehicle_id": 6, "start_node": 0, "end_node": 0, "capacity": 100, "shift_start": 0, "shift_end": 600},
    {"vehicle_id": 7, "start_node": 0, "end_node": 0, "capacity": 20, "shift_start": 0, "shift_end": 600},
    {"vehicle_id": 8, "start_node": 0, "end_node": 0, "capacity": 10, "shift_start": 0, "shift_end": 600},
]


vehicles_df = pd.DataFrame(vehicles)

### Wynik

In [20]:
from milp_model_batch import solve_vrp
from milp_visualize_batch import visualize_routes

batch_size=10 # ile próbek ma jeden batch

### uruchomienie solvera
results = solve_vrp(nodes_df, requests_df, vehicles_df, time_matrix, batch_size)

if results["status"] != "Optimal":
    print("Brak tras do wizualizacji — solver nie znalazł rozwiązania.")
else:
    print("Znaleziono rozwiązanie MILP.")


#zapis wyników do jendego folderu indeksowanego datą
RESULTS_ROOT = Path("results")

timestamp = datetime.now().strftime("%d_%m_%Y_%H-%M")
RUN_DIR = RESULTS_ROOT / timestamp
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Folder symulacji:", RUN_DIR)


#zapis request i vehicles do tego folderu 
requests_path = RUN_DIR / "requests.csv"
vehicles_path = RUN_DIR / "vehicles.csv"

requests_df.to_csv(requests_path, index=False, encoding="utf-8-sig")
vehicles_df.to_csv(vehicles_path, index=False, encoding="utf-8-sig")



#zapis wyniku
results["selected_arcs"].to_csv(
    RUN_DIR / "milp_selected_arcs.csv",
    index=False,
    encoding="utf-8-sig"
)

results["routes"].to_csv(
    RUN_DIR / "milp_routes.csv",
    index=False,
    encoding="utf-8-sig"
)

results["lab_assignments"].to_csv(
    RUN_DIR / "milp_lab_assignments.csv",
    index=False,
    encoding="utf-8-sig"
)

results["load"].to_csv(
    RUN_DIR / "milp_load.csv",
    index=False,
    encoding="utf-8-sig"
)

results["route_load"].to_csv(
    RUN_DIR / "milp_route_load.csv",
    index=False,
    encoding="utf-8-sig"
)

results["travel_time_summary"].to_csv(
    RUN_DIR / "milp_travel_time_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

results["total_route_time"].to_csv(
    RUN_DIR / "milp_total_route_time.csv",
    index=False,
    encoding="utf-8-sig"
)


print("Zapisano wyniki solvera w:", RUN_DIR)


#wizualizacja
if results["status"] == "Optimal":
    routes_df = results["routes"]

    routes_dict = (
        routes_df
        .sort_values(["vehicle_id", "stop_order"])
        .groupby("vehicle_id")["node"]
        .apply(list)
        .to_dict()
    )

    # ============================================================
    # 6. WIZUALIZACJA TRAS I ZAPIS MAPY
    # ============================================================

    summary = visualize_routes(
        nodes_df=nodes_df,
        routes=routes_dict,
        G=G,
        OUT_DIR=RUN_DIR,
        map_filename="milp_routes_map.html"
    )

    summary.to_csv(
        RUN_DIR / "route_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("Mapa i podsumowanie zapisane w:", RUN_DIR)

# ============================================================
# 7. PODSUMOWANIE
# ============================================================

print("\n=== PODSUMOWANIE SYMULACJI ===")
print("Folder:", RUN_DIR)
print("Objective:", results["objective"])
print("Status:", results["status"])

Znaleziono rozwiązanie MILP.
Folder symulacji: results\16_04_2026_08-04
Zapisano wyniki solvera w: results\16_04_2026_08-04

Zapisano mapę: C:\Users\annad\Desktop\operations\results\16_04_2026_08-04\milp_routes_map.html
Mapa i podsumowanie zapisane w: results\16_04_2026_08-04

=== PODSUMOWANIE SYMULACJI ===
Folder: results\16_04_2026_08-04
Objective: 297.33387928992306
Status: Optimal
